# 03 — EfficientNet-B0 Training

Fine-tunes a pretrained **EfficientNet-B0** on the mel-spectrogram images
produced in notebook `pipeline.ipynb`.

| Stage | Details |
|---|---|
| Model | EfficientNet-B0, ImageNet weights |
| Input | 224 × 224 RGB PNG spectrograms |
| Split | stratified 70 / 15 / 15 (train / val / test) |
| Optimiser | Adam + cosine LR annealing |
| Tracking | MLflow (local `mlruns/`) |
| Checkpoint | `models/best_model.pt` (best val loss) |

> **Colab**: make sure the runtime is set to **GPU** (Runtime → Change runtime type → T4 GPU).

## 0 · Colab setup
_Skip this section when running locally — jump straight to section 1._

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

In [ ]:
# ── Mount Google Drive (optional — needed only to persist data / models) ──────
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust the path below to your Drive folder if you use one
    # DRIVE_ROOT = '/content/drive/MyDrive/bird-acoustics'
    print('Drive mounted.')
else:
    print('Local environment — Drive not mounted.')

In [ ]:
if IN_COLAB:
    %pip install -q mlflow scikit-learn tqdm torchvision librosa pyyaml pillow
    print('Dependencies installed.')

In [ ]:
if IN_COLAB:
    import os
    # Clone the repo (replace with your fork URL if needed)
    REPO_URL    = 'https://github.com/danort92/bird-acoustics-classifier.git'
    REPO_BRANCH = 'claude/setup-project-structure-lVRTH'
    REPO_DIR    = '/content/bird-acoustics-classifier'

    if not os.path.exists(REPO_DIR):
        !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull origin {REPO_BRANCH}

    os.chdir(REPO_DIR)
    print('Working directory:', os.getcwd())
    # Inject repo root into Python path so `src` is importable
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

In [ ]:
# ── Local: ensure the repo root is on the path ───────────────────────────────
if not IN_COLAB:
    import os
    from pathlib import Path
    repo_root = Path().resolve().parent  # notebooks/ → project root
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    os.chdir(repo_root)
    print('Working directory:', os.getcwd())

## 1 · Configuration

Defaults are read from `config/default.yaml`.  
Override any parameter by assigning a value (uncomment the relevant lines).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from src.model import TrainingConfig

cfg = TrainingConfig.from_yaml('config/default.yaml')

# ── Optional overrides ────────────────────────────────────────────────────────
# cfg.epochs        = 10
# cfg.batch_size    = 16
# cfg.learning_rate = 3e-4
# cfg.patience      = 5
# cfg.num_workers   = 2   # lower on Colab if you hit DataLoader timeouts

# ── Print resolved config ─────────────────────────────────────────────────────
print(f"{'─'*52}")
print(f"  Model          : {cfg.model_name}")
print(f"  Epochs         : {cfg.epochs}  (patience={cfg.patience})")
print(f"  Batch size     : {cfg.batch_size}")
print(f"  Learning rate  : {cfg.learning_rate}")
print(f"  Val / Test     : {cfg.val_split} / {cfg.test_split}")
print(f"  Image size     : {cfg.img_size[0]}×{cfg.img_size[1]} px")
print(f"  Processed dir  : {cfg.processed_dir}")
print(f"  Output dir     : {cfg.output_dir}")
print(f"  MLflow URI     : {cfg.tracking_uri}")
print(f"{'─'*52}")

## 2 · Dataset overview

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

from src.model import BirdDataset

ds = BirdDataset(cfg.processed_dir)
print(f'Total images : {len(ds)}')
print(f'Classes      : {ds.num_classes}')

# Per-class counts
from collections import Counter
counts = Counter(label for _, label in ds.samples)
names  = [ds.classes[i] for i in sorted(counts)]
vals   = [counts[i] for i in sorted(counts)]

fig, ax = plt.subplots(figsize=(max(8, ds.num_classes * 0.7), 4))
bars = ax.bar(range(len(names)), vals, color='steelblue', edgecolor='white')
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.replace('_', '\n') for n in names], fontsize=7)
ax.set_ylabel('Images')
ax.set_title(f'Image distribution — {ds.num_classes} species, {len(ds)} total')
ax.bar_label(bars, fontsize=7, padding=2)
fig.tight_layout()
plt.show()

## 3 · Training

In [ ]:
import torch
from src.model import BirdTrainer

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}'
      + (f' — {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else ''))

trainer   = BirdTrainer(cfg)
best_path, history = trainer.train()

print(f'\nBest model saved to: {best_path}')
print(f'Test accuracy       : {history["test_acc"]:.3f}')

## 4 · Training curves

In [ ]:
epochs_ran = list(range(1, len(history['train_loss']) + 1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(epochs_ran, history['train_loss'], label='train')
axes[0].plot(epochs_ran, history['val_loss'],   label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Accuracy
axes[1].plot(epochs_ran, history['train_acc'], label='train')
axes[1].plot(epochs_ran, history['val_acc'],   label='val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1)
axes[1].legend()

# Learning rate
axes[2].plot(epochs_ran, history['lr'], color='green')
axes[2].set_title('Learning rate')
axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log')

fig.suptitle(f'Training curves — best val_loss={min(history["val_loss"]):.4f}', fontsize=12)
fig.tight_layout()
plt.show()

## 5 · Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.ticker as ticker

y_true, y_pred = trainer.evaluate(best_path)

cm    = confusion_matrix(y_true, y_pred)
cm_n  = cm.astype('float') / cm.sum(axis=1, keepdims=True)  # row-normalised
n_cls = len(trainer.classes)
short = [c.replace('_', '\n') for c in trainer.classes]

fig, ax = plt.subplots(figsize=(max(10, n_cls * 0.65), max(8, n_cls * 0.55)))
im = ax.imshow(cm_n, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(n_cls))
ax.set_yticks(range(n_cls))
ax.set_xticklabels(short, fontsize=7, rotation=90)
ax.set_yticklabels(short, fontsize=7)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Normalised confusion matrix — test acc={history["test_acc"]:.3f}')

for i in range(n_cls):
    for j in range(n_cls):
        val = cm_n[i, j]
        if val > 0.01:
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=6, color='white' if val > 0.5 else 'black')

fig.tight_layout()
plt.show()

print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=trainer.classes, digits=3))

## 6 · Checkpoint summary & download

In [ ]:
import torch

ckpt = torch.load(best_path, map_location='cpu')
print(f"Checkpoint : {best_path}")
print(f"  Epoch    : {ckpt['epoch']}")
print(f"  Val loss : {ckpt['val_loss']:.4f}")
print(f"  Val acc  : {ckpt['val_acc']:.3f}")
print(f"  Classes  : {ckpt['num_classes']}")
print(f"  Img size : {ckpt['img_size']}")
print(f"  Model    : {ckpt['model_name']}")

if IN_COLAB:
    from google.colab import files
    files.download(str(best_path))
    print('\nDownload started.')
else:
    print('\nTo use the model for inference:')
    print('  from src.model import predict')
    print(f'  predict("path/to/spec.png", "{best_path}")')

## 7 · MLflow UI

Run the cell below to start the MLflow tracking server and open it in a new browser tab.  
On Colab, use [ngrok](https://ngrok.com/) or the built-in tunnel.

In [ ]:
import mlflow
import yaml

with open('config/default.yaml') as f:
    _cfg = yaml.safe_load(f)

tracking_uri = _cfg['mlflow']['tracking_uri']
exp_name     = _cfg['mlflow']['experiment_name']

mlflow.set_tracking_uri(tracking_uri)
client = mlflow.tracking.MlflowClient()

exp = client.get_experiment_by_name(exp_name)
if exp:
    runs = client.search_runs(exp.experiment_id, order_by=['metrics.val_loss ASC'], max_results=5)
    print(f'Top 5 runs for "{exp_name}":\n')
    for r in runs:
        m = r.data.metrics
        p = r.data.params
        print(
            f"  run_id={r.info.run_id[:8]}…  "
            f"val_loss={m.get('val_loss', float('nan')):.4f}  "
            f"val_acc={m.get('val_acc', float('nan')):.3f}  "
            f"test_acc={m.get('test_acc', float('nan')):.3f}  "
            f"epochs={p.get('epochs')}  lr={p.get('learning_rate')}"
        )
else:
    print(f'No experiment named "{exp_name}" found in {tracking_uri}.')

print(f'\nTo open MLflow UI locally: mlflow ui --backend-store-uri {tracking_uri}')